> **Chiến dịch hiện tại: v15d3 phân đoạn câu để dịch.** Input chữ thường, không dấu câu, lookahead 1/2 từ; ba model VI/EN/ZH riêng. Không dùng Qwen. Worker crawl/train chạy nền trên Colab và sao lưu trực tiếp lên Drive. Phần thử nghiệm V2 bên dưới là lịch sử; không chạy toàn bộ notebook để khởi động lại chiến dịch đang chạy.

# Turn detector V2 — train trên Colab L4

Mục tiêu: **accuracy ≥90% và F1 EOT ≥90%** trên holdout đã khóa, với ngưỡng chọn từ dev.
Kết quả bản thử 24/10: holdout accuracy **93,81%**, F1 EOT **90,62%** trên bộ nhãn sẵn. Xem phần kết quả cuối notebook và [hồ sơ thử nghiệm](EXPERIMENT_TRIAL_20261024.md).

- Hướng train: mDeBERTa đa ngôn ngữ, AdamW, warmup/cosine, early stopping, gradient checkpointing, FP32/TF32 mặc định.
- Hướng data: crawl có nguồn → Qwen 4B chạy trên GPU → loại mơ hồ/bất đồng → silver chỉ thêm vào train với weight 0,5.
- Cùng chuẩn tiền xử lý khi train, dự đoán và export. Test cũ đã được xem ở baseline, không dùng để chốt mốc mới.

Trong VS Code chọn kernel **Colab GPU L4**. Chạy các ô từ trên xuống.
Mặc định notebook chạy smoke 3 bước để kiểm tra hệ thống; bật các cờ ở ô cấu hình để train đầy đủ.

Legacy test regression: accuracy **93,47%**, F1 EOT **88,68%**; F1 is still below 90% on this set.


## Kết nối Drive để lưu dữ liệu và checkpoint
Đã cấu hình lưu trực tiếp từ Colab vào `MyDrive/turn_detect_backups/translation_segmenter_v15d_20261026`. Worker tự sao lưu corpus, model, optimizer và kết quả test; máy Windows chỉ nhận báo cáo nhỏ. Sau khi đổi VM, chạy ô mount này và tự cấp quyền Drive. Giữ kernel hiện tại khi worker đang chạy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Drive đã gắn. Worker sao lưu sẽ tự tiếp tục khi nhận diện đúng thư mục.")


In [ ]:
!nvidia-smi

In [ ]:
import json, os, pathlib, platform, subprocess, sys
import torch
assert pathlib.Path('/content').is_dir() and sys.platform.startswith('linux'), 'Chọn kernel Colab, không chọn Python Windows.'
assert torch.cuda.is_available(), 'Runtime chưa có CUDA GPU.'
print(json.dumps({'python':platform.python_version(), 'torch':torch.__version__,
    'gpu':torch.cuda.get_device_name(0), 'cuda':torch.cuda.is_available(),
    'vram_gib':round(torch.cuda.get_device_properties(0).total_memory/2**30,2)}, indent=2))


In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
ROOT = Path('/content/turn_detect_v2')
DATA = ROOT / 'data_v2/frozen'
# Đổi RUNS_ROOT sang thư mục Drive đã mount nếu cần giữ checkpoint qua lần ngắt runtime.
RUNS_ROOT = ROOT / 'runs'
RUN_TAG = datetime.now(timezone(timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
BASE_MODEL = 'microsoft/mdeberta-v3-base'
QWEN_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
RUN_SMOKE = True
RUN_DATA_ROUTE = True
RUN_FULL_TRAIN = False
RUN_FINAL_HOLDOUT = False
EXPERIMENTS = ['gold']  # So sánh hai hướng: ['gold', 'gold_silver'], cùng split/seed.
EXISTING_RUN_DIRS = []  # Có thể đặt các thư mục run đã train để chọn/evaluate sau khi reconnect.
QWEN_LIMIT = 4  # Smoke; tăng lên khi chuẩn bị augmentation cho train đầy đủ.
print({'root':str(ROOT), 'run_tag':RUN_TAG, 'full_train':RUN_FULL_TRAIN, 'experiments':EXPERIMENTS})


In [ ]:
import base64, hashlib, io, zipfile
package = base64.b64decode('...')
assert hashlib.sha256(package).hexdigest() == 'e1944613d57fb3d9d8fa6ce1b5a045ab7bf9dc971862f6c769f242f8970ede14'
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(package)) as archive:
    for name in archive.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'  # Public models; no Colab Secrets popup in VS Code.
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
def run_module(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env={**os.environ, 'OMP_NUM_THREADS':'2'}) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
print('Mã nguồn đã giải nén và kiểm tra SHA-256:', ROOT)


In [ ]:
import importlib.metadata
required = {'transformers':'5.20.1','datasets':'4.8.5','accelerate':'1.15.0',
            'sentencepiece':'0.2.2','safetensors':'0.8.0','onnx':'1.19.1','onnxruntime':'1.23.2'}
installed = {}
for name in required:
    try: installed[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: installed[name] = None
if any(installed[name] != version for name,version in required.items()):
    subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements-colab.txt')],check=True)
print({name:importlib.metadata.version(name) for name in required})


## Dữ liệu và split cố định

Code và dữ liệu đã được đồng bộ tới runtime hiện tại. Khi dùng runtime mới, chạy `python tools/sync_colab.py`
ở Windows sau khi kết nối lại kernel; hoặc upload `colab/turn_detect_v2_bundle.zip` vào `/content`.
ZIP đầy đủ chứa cả `train.jsonl`, `test.jsonl` và split đã khóa.

Không chia ngẫu nhiên từng đoạn cắt của cùng câu nguồn. Mã ưu tiên `source_id`; với dữ liệu cũ,
nhóm được suy từ câu EOT và prefix theo thứ tự, rồi nối các prefix dài. Đây là suy đoán vì dữ liệu cũ thiếu ID hội thoại.
Các nhóm trùng test cũ và nhãn mâu thuẫn được cách ly; không đổi nhãn dev/holdout bằng Qwen.


In [ ]:
if not (ROOT/'train.jsonl').exists() or not (ROOT/'test.jsonl').exists():
    bundle = Path('/content/turn_detect_v2_bundle.zip')
    if bundle.exists():
        with zipfile.ZipFile(bundle) as archive:
            for name in archive.namelist():
                assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
            archive.extractall(ROOT)
assert (ROOT/'train.jsonl').exists() and (ROOT/'test.jsonl').exists(), 'Upload ZIP đầy đủ vào /content hoặc chạy tools/sync_colab.py từ Windows.'
run_module('turn_v2.prepare', '--train', ROOT/'train.jsonl', '--legacy-test', ROOT/'test.jsonl', '--output', DATA)
manifest = json.loads((DATA/'manifest.json').read_text())
print(json.dumps(manifest['splits'], ensure_ascii=False, indent=2))


In [ ]:
tested = subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests')],cwd=ROOT,
                        check=True,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(tested.stdout)


## Hướng dữ liệu: crawl + Qwen

Bật `RUN_DATA_ROUTE=True` ở ô cấu hình. Qwen 4B dùng GPU nên chạy riêng trước encoder.
Câu gốc và đoạn cắt đều là ứng viên chưa có nhãn; Qwen được phép chọn `AMBIGUOUS`.
Hai góc hỏi cùng model vẫn có sai số tương quan, nên nhãn được giữ là silver và không thay thế gold người duyệt.
`audit.jsonl` giữ cả hai câu trả lời; source URL được giữ để truy nguồn.
Wikipedia là nguồn văn xuôi ban đầu, chưa đại diện đầy đủ cho hội thoại thật.


In [ ]:
SILVER = None
if RUN_DATA_ROUTE:
    candidates = ROOT/'data_v2'/f'candidates_{RUN_TAG}.jsonl'
    run_module('turn_v2.crawl','--output',candidates,'--max-candidates',QWEN_LIMIT)
    teacher_dir = ROOT/'data_v2'/f'qwen_{RUN_TAG}'
    run_module('turn_v2.label_qwen','--input',candidates,'--output',teacher_dir,
               '--model',QWEN_MODEL,'--limit',QWEN_LIMIT)
    labeling = json.loads((teacher_dir/'labeling_report.json').read_text())
    assert labeling['accepted'] > 0, 'Qwen chưa tạo nhãn đủ tiêu chí; xem audit trước khi chạy hướng silver.'
    SILVER = teacher_dir/'silver.jsonl'
else:
    print('Bỏ qua crawl/label trong lượt này; để bật: RUN_DATA_ROUTE=True.')


## Smoke test: train → save/reload → ONNX parity

Smoke có 32 mẫu train nhãn sẵn (thêm tối đa 8 silver khi bật hướng data), 16 mẫu dev và 3 bước optimizer, không mở holdout.
Các điểm số smoke không dùng để đánh giá chất lượng hoặc xác nhận đạt 90%.
Export FP32 kiểm tra chênh xác suất PyTorch/ONNX; INT8 cần một lượt kiểm chứng riêng.


In [ ]:
SMOKE_DIR = RUNS_ROOT/f'smoke_{RUN_TAG}'
if RUN_SMOKE:
    smoke_arguments = ['--data',DATA,'--model',BASE_MODEL,'--output',SMOKE_DIR,
                       '--smoke','--batch-size',4,'--max-steps',3,'--export']
    if SILVER is not None:
        smoke_arguments += ['--silver',SILVER,'--legacy-test',ROOT/'test.jsonl']
    run_module('turn_v2.train',*smoke_arguments)
    smoke_report = json.loads((SMOKE_DIR/'training_report.json').read_text())
    assert smoke_report['optimizer_steps'] == 3 and not smoke_report['holdout_evaluated']
    assert smoke_report['export']['passed']
    print('SMOKE PASSED: GPU train, checkpoint reload và ONNX parity.')


## Train đầy đủ và so sánh hai hướng

Bật `RUN_FULL_TRAIN=True`. Mặc định train 3 epoch, learning rate encoder `2e-5`, head `1e-4`,
batch 16 × tích lũy 2, FP32/TF32, warmup 6%, cosine, gradient clipping và early stopping.
FP32 là cấu hình đã được kiểm tra; BF16 từng tạo đầu ra không hữu hạn trong smoke trên runtime này.

Chạy `EXPERIMENTS=['gold','gold_silver']` cùng `RUN_DATA_ROUTE=True` để so sánh cùng split/seed.
Chọn checkpoint và ngưỡng bằng dev; chưa mở holdout ở bước này. Khi GPU hết bộ nhớ, giảm batch và tăng accumulation để giữ batch hiệu dụng 32.
Resume dùng cùng thư mục run: thêm `--resume <run>/checkpoints/checkpoint-N` khi chạy CLI; code kiểm tra data/model/seed/settings/source trước khi resume.


In [ ]:
completed_runs = []
if RUN_FULL_TRAIN:
    from huggingface_hub import model_info
    base_revision = model_info(BASE_MODEL,token=False).sha
    assert EXPERIMENTS and set(EXPERIMENTS) <= {'gold','gold_silver'}
    assert 'gold_silver' not in EXPERIMENTS or SILVER is not None, 'Bật RUN_DATA_ROUTE hoặc đặt SILVER trỏ file silver đã gán nhãn.'
    for variant in EXPERIMENTS:
        output = RUNS_ROOT/f'{variant}_{RUN_TAG}'
        arguments = ['--data',DATA,'--model',BASE_MODEL,'--output',output,
                     '--revision',base_revision,'--batch-size',16,'--accumulation',2,'--epochs',3,'--seed',42,'--export']
        if variant == 'gold_silver':
            arguments += ['--silver',SILVER,'--legacy-test',ROOT/'test.jsonl']
        run_module('turn_v2.train',*arguments)
        report = json.loads((output/'training_report.json').read_text())
        completed_runs.append((output, report['dev_selected_threshold']))
    print(json.dumps([{'run':str(path), **metrics} for path,metrics in completed_runs],indent=2))
else:
    for existing in EXISTING_RUN_DIRS:
        output = Path(existing)
        report = json.loads((output/'training_report.json').read_text())
        assert report['mode'] == 'FULL_DEV_SELECTION'
        completed_runs.append((output,report['dev_selected_threshold']))
    print('Để train đầy đủ: RUN_FULL_TRAIN=True. Chưa có kết quả 90% từ smoke.')


## Chốt một model và đánh giá holdout một lần

Bật `RUN_FINAL_HOLDOUT=True` sau khi chạy train đầy đủ. Chọn model có `min(accuracy,F1 EOT)` cao nhất trên dev.
Nếu dev chưa đạt cả hai mốc 90%, holdout được giữ kín. Nếu đã mở holdout, không dùng điểm đó để chọn model/threshold tiếp theo.
Kết quả chỉ xác nhận trên nhãn được cung cấp và cách nhóm heuristic; muốn khẳng định trên hội thoại thật cần gold mới do người duyệt.


In [ ]:
if RUN_FINAL_HOLDOUT:
    assert completed_runs, 'Train đầy đủ trước khi mở holdout.'
    chosen, dev_metrics = max(completed_runs,key=lambda item:item[1]['joint_score'])
    run_module('turn_v2.evaluate','--model',chosen/'best','--data',DATA,'--output',chosen/'final')
    final = json.loads((chosen/'final/holdout_report.json').read_text())
    print(final['status'],final['metrics'])
else:
    print('Holdout chưa được mở.')


In [ ]:
if RUN_SMOKE and (SMOKE_DIR/'onnx/parity_report.json').exists():
    run_module('turn_v2.predict','--model',SMOKE_DIR/'onnx','--text','tôi muốn đặt vé đi hà nội')
print('Checkpoint/report:', RUNS_ROOT)
print('Tải model best và report qua bảng Colab Contents của VS Code. /content sẽ mất khi runtime bị xóa.')


## Crawl và dự đoán ngay, rồi Qwen xét một số mẫu

Luồng này ghi từng dự đoán khi crawl được mẫu, không tự cập nhật trọng số và không phải RL.
Qwen xét cả mẫu gần ngưỡng lẫn mẫu EOT có xác suất cao trên đoạn cắt; nhãn chấp nhận vẫn là silver.
Đặt `PREDICT_MODEL_DIR` trỏ model train đầy đủ khi có. Nếu chỉ có checkpoint smoke, output được đánh dấu chạy thử.
Muốn RL sau này cần thêm state/action/reward cho việc chờ tiếp hay kết thúc lượt, với tham chiếu thời điểm kết thúc thật.


In [ ]:
from datetime import datetime, timedelta, timezone
TRIAL_ONNX = ROOT/'runs/trial_gold_20261024_093324/onnx'
COMPACT_ONNX = ROOT/'models/turn_v2_compact_20261024/onnx'
DISTILLED_ONNX = ROOT/'models/turn_v2_distilled_min_20261024/onnx'
INT4_ONNX = ROOT/'models/turn_v2_distilled_int4_20261024/onnx'
QAT_ONNX = ROOT/'models/turn_v2_distilled_qat4_20261024/onnx'
PREDICT_MODEL_DIR = QAT_ONNX if QAT_ONNX.is_dir() else (INT4_ONNX if INT4_ONNX.is_dir() else (DISTILLED_ONNX if DISTILLED_ONNX.is_dir() else (COMPACT_ONNX if COMPACT_ONNX.is_dir() else (TRIAL_ONNX if TRIAL_ONNX.is_dir() else None))))  # Ví dụ: ROOT/'runs/gold_v1/onnx'
if PREDICT_MODEL_DIR is None:
    if globals().get('completed_runs'):
        chosen = max(completed_runs,key=lambda item:item[1]['joint_score'])[0]
        PREDICT_MODEL_DIR = chosen/'onnx'
    elif globals().get('SMOKE_DIR') and (SMOKE_DIR/'onnx').exists():
        PREDICT_MODEL_DIR = SMOKE_DIR/'onnx'
    else:
        raise ValueError('Đặt PREDICT_MODEL_DIR trỏ một checkpoint đã có.')
CRAWL_TAG = datetime.now(timezone(timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
CRAWL_OUTPUT = ROOT/'outputs'/f'crawl_predict_{CRAWL_TAG}'
contract = json.loads((Path(PREDICT_MODEL_DIR)/'turn_detector_config.json').read_text())
arguments = ['--model',PREDICT_MODEL_DIR,'--output',CRAWL_OUTPUT,'--limit',12,
             '--protected-data',DATA,'--legacy-test',ROOT/'test.jsonl','--review-budget',4,
             '--qwen-review','--teacher-model',QWEN_MODEL]
if contract.get('smoke_only'):
    print('Checkpoint SMOKE: chỉ kiểm tra pipeline, chưa phải model đạt 90%.')
    arguments += ['--allow-smoke']
run_module('turn_v2.crawl_predict',*arguments)
from IPython.display import HTML, display
display(HTML((CRAWL_OUTPUT/'results.html').read_text()))
print('Dự đoán và audit:',CRAWL_OUTPUT)


Nguồn chính: [mDeBERTa V3](https://huggingface.co/microsoft/mdeberta-v3-base),
[Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507),
[Transformers Trainer](https://huggingface.co/docs/transformers/v5.20.0/main_classes/trainer),
[Colab extension](https://github.com/googlecolab/colab-vscode).


# Bản train thử ngày 24/10/2026

Đã đạt **accuracy 93,81% và F1 EOT 90,62%** trên 17.099 mẫu holdout đã khóa. Ngưỡng 0,5 và checkpoint được chọn trên dev trước khi mở holdout. Đây là kết quả trên nhãn dữ liệu cung cấp, với cách gom nhóm nguồn bằng heuristic; chưa phải kết quả kiểm định hội thoại thực do người rà soát.

| Tập đánh giá | Số mẫu | Accuracy | F1 EOT | Precision EOT | Recall EOT |
| --- | ---: | ---: | ---: | ---: | ---: |
| Dev | 17.833 | 93,78% | 90,55% | 86,72% | 94,73% |
| Holdout cuối | 17.099 | 93,81% | 90,62% | 86,64% | 94,99% |
| Test cũ, kiểm tra hồi quy | 4.183 | 93,47% | 88,68% | 89,98% | 87,41% |

**F1 EOT trên bộ test cũ chưa đạt 90%.** Bộ này được đo sau khi đã khóa model/ngưỡng; không dùng kết quả để đổi checkpoint hay ngưỡng. Model cũ trên cùng test đạt accuracy 65,03%, F1 EOT 58,94%; lỗi cắt sớm giảm từ 1.290 xuống 119, EOT bị bỏ lỡ giảm từ 173 xuống 154. Đây là kiểm tra hồi quy, vì test cũ đã được xem trước đợt train này. [Receipt và dự đoán từng mẫu](outputs/training_trial_20261024/legacy_regression/report.json).

Holdout có 789 lần cắt sớm và 270 EOT bị bỏ lỡ. Holdout đã được đánh dấu sử dụng ở `data_v2/frozen/holdout_consumed.json`; không dùng tập này để chọn tiếp model/ngưỡng khác.


[Review HTML](outputs/training_trial_20261024/review.html) | [Full experiment record](EXPERIMENT_TRIAL_20261024.md)


In [ ]:
import json
from pathlib import Path
IS_COLAB = Path('/content/turn_detect_v2').is_dir()
TRIAL_RUN = Path('/content/turn_detect_v2/runs/trial_gold_20261024_093324')
TRIAL_RECEIPTS = TRIAL_RUN if IS_COLAB else Path('outputs/training_trial_20261024')
TRIAL_FINAL = TRIAL_RUN/'final_holdout' if IS_COLAB else TRIAL_RECEIPTS
trial_train = json.loads((TRIAL_RECEIPTS/'training_report.json').read_text(encoding='utf-8'))
trial_final = json.loads((TRIAL_FINAL/'holdout_report.json').read_text(encoding='utf-8'))
trial_legacy = json.loads((TRIAL_RECEIPTS/'legacy_regression/report.json').read_text(encoding='utf-8'))
PREDICT_MODEL_DIR = TRIAL_RUN/'onnx' if IS_COLAB else Path('models/turn_v2_trial_20261024/onnx')
assert trial_final['status'] == 'TARGET_MET'
assert trial_train['export']['passed']
print(json.dumps({'run':'trial_gold_20261024_093324','optimizer_steps':trial_train['optimizer_steps'],
    'selected_checkpoint':'checkpoint-1750','dev':trial_train['dev_selected_threshold'],
    'holdout':trial_final['metrics'],'legacy_test_regression':trial_legacy['metrics'],'prediction_model':str(PREDICT_MODEL_DIR),
    'scope':trial_final['scope']},ensure_ascii=False,indent=2))


# Bản nén ngày 24/10/2026

Đã đạt ngân sách **≤1/10 dung lượng ONNX gốc**: ONNX **108.202 MB**, toàn bộ gói **109.609 MB**, so với 1.116,285 MB ban đầu.

Trên **17.833 mẫu dev**, accuracy **93.792%**, F1 EOT **90.523%**. Đây là kết quả chọn bản nén trên dev; **chưa chấm holdout độc lập mới**. Holdout lần train trước đã dùng và không được mở lại để chọn bản nén.

Thu gọn vocab từ train và lượng tử hóa INT8 trọng số; giữ 12 lớp, activation và phép tính FP32. Không train lại, không distillation/RL. Mục tiêu 1/10 là file lưu trữ, không phải RAM, số tham số hay tốc độ.

Ô dưới chạy độc lập, kiểm tra SHA256 và suy luận với bản nén trên Windows hoặc Colab đã đồng bộ. [Review chi tiết](outputs/compression_20261024/review.html) · [Experiment record](EXPERIMENT_COMPRESSION_20261024.md)


In [ ]:
import hashlib, json, sys
from pathlib import Path
COMPACT_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
sys.path.insert(0,str(COMPACT_ROOT))
from turn_v2.predict import TurnDetectorV2
COMPACT_MANIFEST = json.loads((COMPACT_ROOT/'models/turn_v2_compact_20261024/model_manifest.json').read_text(encoding='utf-8'))
PREDICT_MODEL_DIR = COMPACT_ROOT/'models/turn_v2_compact_20261024/onnx'
assert COMPACT_MANIFEST['status']=='SIZE_AND_FULL_DEV_TARGET_MET'
for filename, identity in COMPACT_MANIFEST['files'].items():
    file_path = PREDICT_MODEL_DIR/filename
    assert file_path.stat().st_size==identity['bytes']
    assert hashlib.file_digest(file_path.open('rb'),'sha256').hexdigest()==identity['sha256']
compact_detector = TurnDetectorV2(PREDICT_MODEL_DIR)
demo_texts = ['tôi muốn đặt vé đi hà nội', 'tôi muốn đặt vé đi', 'cảm ơn bạn']
compact_demos = [{'text':text,**compact_detector.predict(text)} for text in demo_texts]
print(json.dumps({'deployment_MB':COMPACT_MANIFEST['deployment_bytes']/1e6,
    'dev':COMPACT_MANIFEST['dev_metrics'],'scope':COMPACT_MANIFEST['quality_scope'],
    'model':str(PREDICT_MODEL_DIR),'hashes_verified':True,'demos':compact_demos},ensure_ascii=False,indent=2))


# Distillation: student nhỏ nhất đã thử đạt mục tiêu

**GRU 96, embedding 64, 1 lớp**, 293,090 tham số triển khai. ONNX **328.761 KB**, cả thư mục model/tokenizer/runner/requirements **555.479 KB** (không tính thư viện runtime đã cài), nhỏ hơn **197.3 lần** gói teacher 109,6 MB.

Trên **17.833 mẫu dev**: accuracy **93.562%**, F1 EOT **90.074%**, ngưỡng **0.545**. Đã thử 6 lượt train/refinement và 18 biến thể lưu trọng số. Đây là bản nhỏ nhất **trong các bản đã thử** đạt cả hai chỉ số, không chứng minh mức nhỏ nhất tuyệt đối.

Student học từ nhãn train, logits và đặc trưng teacher cố định trên GPU L4. **Dev được dùng chọn bản/ngưỡng; holdout đã dùng không chấm lại.** F1 chỉ nhỉnh hơn 90% một ít, chưa bảo đảm trên hội thoại thực/câu ngắn.

Ô dưới kiểm tra hashes và suy luận bằng runner độc lập chỉ cần NumPy, Tokenizers và ONNX Runtime; không cần PyTorch/Transformers cho suy luận. [Model và hướng dẫn](models/turn_v2_distilled_min_20261024/README.md) · [Review](outputs/distillation_20261024/review.html) · [Hồ sơ](EXPERIMENT_DISTILLATION_20261024.md)


In [ ]:
import hashlib, importlib.util, json, sys
from pathlib import Path
DISTILL_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
DISTILL_DIR = DISTILL_ROOT/'models/turn_v2_distilled_min_20261024'
distill_manifest = json.loads((DISTILL_DIR/'model_manifest.json').read_text(encoding='utf-8'))
PREDICT_MODEL_DIR = DISTILL_DIR/'onnx'
for filename, identity in distill_manifest['files'].items():
    file_path=PREDICT_MODEL_DIR/filename
    assert file_path.stat().st_size==identity['bytes']
    assert hashlib.file_digest(file_path.open('rb'),'sha256').hexdigest()==identity['sha256']
runtime_spec=importlib.util.spec_from_file_location('tiny_portable_runtime',PREDICT_MODEL_DIR/'run_model.py')
tiny_runtime=importlib.util.module_from_spec(runtime_spec)
previous_bytecode=sys.dont_write_bytecode
sys.dont_write_bytecode=True
try:runtime_spec.loader.exec_module(tiny_runtime)
finally:sys.dont_write_bytecode=previous_bytecode
tiny_detector=tiny_runtime.TinyOnnxDetector(PREDICT_MODEL_DIR)
tiny_texts=['hà nội là thủ đô và là một trong những trung tâm văn hóa lớn của việt nam','tôi muốn đặt vé đi','cảm ơn bạn']
print(json.dumps({'onnx_KB':distill_manifest['onnx_bytes']/1000,'deployment_KB':distill_manifest['deployment_bytes']/1000,
    'dev':distill_manifest['dev_metrics'],'scope':distill_manifest['quality_scope'],
    'hashes_verified':True,'model':str(PREDICT_MODEL_DIR),
    'demos':[{'text':text,**tiny_detector.predict(text)} for text in tiny_texts]},ensure_ascii=False,indent=2))


## Student INT4 đã kiểm tra

Trọng số ma trận **INT4**, nhóm32, scale FP16; bias/norm và phép tính FP32. **ONNX 185,933 KB**, cả model/tokenizer/runner **412,682 KB** (thư viện đã cài không tính). INT8 đối chứng tương ứng 328,761 / 555,479 KB.

Đủ **17.833 dev**: accuracy **93,4335%**, F1 EOT **90,0620%**, ngưỡng hiệu chỉnh **0,430**. Nhóm32 nhỏ nhất đạt trong bốn cấu hình đã chấm. Dev dùng chọn cấu hình và ngưỡng; không phải holdout độc lập. Giữ ngưỡng INT8 0,545 thì F1 INT4 chỉ89,8236%. False EOT tăng744→864, missed EOT giảm404→307.

Ô dưới dùng runner độc lập, kiểm tra hashes và chạy ba ví dụ cố định; dự đoán sai trên câu ngắn vẫn được giữ để xem giới hạn. Bản INT8 cũ còn nguyên.

[Model/hướng dẫn](models/turn_v2_distilled_int4_20261024/README.md) · [ZIP](models/turn_v2_distilled_int4_20261024.zip) · [Hồ sơ](EXPERIMENT_INT4_20261024.md)


In [ ]:
import hashlib, importlib.util, json, sys
from pathlib import Path
INT4_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
INT4_DIR = INT4_ROOT/'models/turn_v2_distilled_int4_20261024'
int4_manifest = json.loads((INT4_DIR/'model_manifest.json').read_text(encoding='utf-8'))
PREDICT_MODEL_DIR = INT4_DIR/'onnx'
for filename, identity in int4_manifest['files'].items():
    file_path = PREDICT_MODEL_DIR/filename
    assert file_path.stat().st_size == identity['bytes']
    with file_path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == identity['sha256']
int4_spec = importlib.util.spec_from_file_location('int4_portable_runtime', PREDICT_MODEL_DIR/'run_model.py')
int4_runtime = importlib.util.module_from_spec(int4_spec)
previous_bytecode = sys.dont_write_bytecode
sys.dont_write_bytecode = True
try: int4_spec.loader.exec_module(int4_runtime)
finally: sys.dont_write_bytecode = previous_bytecode
int4_detector = int4_runtime.TinyOnnxDetector(PREDICT_MODEL_DIR)
int4_texts = ['hà nội là thủ đô và là một trong những trung tâm văn hóa lớn của việt nam', 'tôi muốn đặt vé đi', 'cảm ơn bạn']
print(json.dumps({'hashes_verified': True, 'onnx_KB': int4_manifest['onnx_bytes']/1000,
    'deployment_KB': int4_manifest['deployment_bytes']/1000, 'dev': int4_manifest['dev_metrics'],
    'scope': int4_manifest['quality_scope'], 'model': str(PREDICT_MODEL_DIR),
    'demos': [{'text': text, **int4_detector.predict(text)} for text in int4_texts]}, ensure_ascii=False, indent=2))


## Student QAT INT4

Đã chạy QAT thật trên **315.174 train**, GPU L4: giả lập4-bit ở mỗi forward, cập nhật trọng số bằng clipped STE và loss distillation. Scale FP16 giữ nguyên từ train sang export, không fit lại khi nén. Hai lượt QAT nhóm64/32; chọn nhóm**64**, best epoch**4**.

ONNX **177.124 KB**, cả gói **403.973 KB**. Chính ONNX packed INT4 đạt accuracy **93.5457%**, F1 EOT **90.1934%**, ngưỡng **0.470** trên đủ17.833 dev. Dev dùng chọn checkpoint/ngưỡng, chưa có holdout độc lập mới. Trọng số4-bit, activations/phép tính FP32.

[Model/hướng dẫn](models/turn_v2_distilled_qat4_20261024/README.md) · [ZIP](models/turn_v2_distilled_qat4_20261024.zip) · [Hồ sơ](EXPERIMENT_QAT_20261024.md). Bản INT8 và PTQ giữ nguyên.


In [ ]:
import hashlib, importlib.util, json, sys
from pathlib import Path
QAT_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
QAT_DIR = QAT_ROOT/'models/turn_v2_distilled_qat4_20261024'
qat_manifest = json.loads((QAT_DIR/'model_manifest.json').read_text(encoding='utf-8'))
PREDICT_MODEL_DIR = QAT_DIR/'onnx'
for filename, identity in qat_manifest['files'].items():
    file_path = PREDICT_MODEL_DIR/filename
    assert file_path.stat().st_size == identity['bytes']
    with file_path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == identity['sha256']
qat_spec = importlib.util.spec_from_file_location('qat_portable_runtime', PREDICT_MODEL_DIR/'run_model.py')
qat_runtime = importlib.util.module_from_spec(qat_spec)
previous_bytecode = sys.dont_write_bytecode
sys.dont_write_bytecode = True
try: qat_spec.loader.exec_module(qat_runtime)
finally: sys.dont_write_bytecode = previous_bytecode
qat_detector = qat_runtime.TinyOnnxDetector(PREDICT_MODEL_DIR)
qat_texts = ['hà nội là thủ đô và là một trong những trung tâm văn hóa lớn của việt nam', 'tôi muốn đặt vé đi', 'cảm ơn bạn']
print(json.dumps({'hashes_verified': True, 'training_method': qat_manifest['training_method'],
    'onnx_KB': qat_manifest['onnx_bytes']/1000, 'deployment_KB': qat_manifest['deployment_bytes']/1000,
    'dev': qat_manifest['dev_metrics'], 'scope': qat_manifest['quality_scope'], 'model': str(PREDICT_MODEL_DIR),
    'demos': [{'text': text, **qat_detector.predict(text)} for text in qat_texts]}, ensure_ascii=False, indent=2))


## Pilot Việt–Anh–Trung: đã train, chưa đạt chất lượng để thay bản Việt

Teacher12lớp đã fine-tune800bước; student GRU128 distill11epoch rồi QAT6epoch. Train71.329 mẫu, dev7.190 mẫu từ Việt frozen-train + MultiWOZ2.2 + CrossWOZ. ONNX729KB, cả gói1,889MB, vocabulary18.711. Trọng số INT4, phép tính FP32.

**Không thay `PREDICT_MODEL_DIR` đang dùng bản Việt.** Student có regression Việt; English/Chinese chưa đạt90%. Nhãn cắt prefix là proxy, không phải semantic gold. Ngoài test nhãn suy ra, đã kiểm tra744turn thật ở LiveKit,2.057điểm dừng đủ200ms; chỉ dùng transcript/timing, không tải audio, không tune trên benchmark.

[Báo cáo tổng](outputs/trilingual_20261024/review.html) · [Player từng prefix và dấu phẩy/chấm](outputs/trilingual_stream_20261024/review.html) · [Hướng dẫn/dữ liệu/model](README_TRILINGUAL_V3.md).

Ô dưới kiểm tra hashes và chạy3ngôn ngữ trên đúng file INT4. Đầu vào dài bị cắt sẽ được đánh dấu; dấu câu tự nó không phải tín hiệu EOT trong model này.


In [ ]:
import hashlib, importlib.util, json, sys
from pathlib import Path
TRI_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
TRI_DIR = TRI_ROOT/'models/turn_v3_trilingual_pilot_20261024'
tri_manifest = json.loads((TRI_DIR/'model_manifest.json').read_text(encoding='utf-8'))
for filename, identity in tri_manifest['files'].items():
    file_path = TRI_DIR/'onnx'/filename
    assert file_path.stat().st_size == identity['bytes']
    with file_path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == identity['sha256']
tri_spec = importlib.util.spec_from_file_location('trilingual_portable', TRI_DIR/'onnx/run_model.py')
tri_runtime = importlib.util.module_from_spec(tri_spec)
previous_bytecode = sys.dont_write_bytecode
sys.dont_write_bytecode = True
try: tri_spec.loader.exec_module(tri_runtime)
finally: sys.dont_write_bytecode = previous_bytecode
tri_detector = tri_runtime.TinyOnnxDetector(TRI_DIR/'onnx')
tri_cases = [('vi', 'tôi muốn đặt một vé tàu đi Hà Nội vào sáng mai'), ('vi', 'tôi muốn đặt một vé tàu đi'),
             ('en', 'I would like to book a flight to London tomorrow'), ('en', 'I would like to book a flight to'),
             ('zh', '我想订一张明天从北京到上海的火车票'), ('zh', '我想订一张从北京到')]
print(json.dumps({'hashes_verified': True, 'status': tri_manifest['status'],
    'onnx_KB': tri_manifest['onnx_bytes']/1000, 'deployment_KB': tri_manifest['deployment_bytes']/1000,
    'proxy_test': tri_manifest['derived_official_test'], 'real_pause_text_only': tri_manifest['external_real_pause_text_only'],
    'vi_full_dev': tri_manifest['vi_full_dev_regression'], 'serving_default_changed': False,
    'demos': [{'text': text, **tri_detector.predict(text, language=lang)} for lang, text in tri_cases]}, ensure_ascii=False, indent=2))


## Streaming V4: đã train/test lại, chưa đạt >90% cả ba tiếng

Colab3×600bước +300bước cân bằng nhãn; local3×200bước. Checkpoint chọn ở vòng2,bước200, đã backup đầy đủ. Pilot audio mới70mẫu: Việt accuracy50,0%/F157,1%; Anh70,0%/66,7%; Trung55,0%/66,7%. Nhãn VAD tự động, không gọi là human gold.204prefix crawl mới chưa có nhãn nên accuracy/F1=null.

[Báo cáo và player](outputs/stream_v4_20261025/review.html) · [Nghe30điểm Việt](outputs/stream_v4_20261025/vi_natural_review.html) · [Hướng dẫn](README_STREAM_V4.md). Ô dưới chỉ kiểm tra hash/đọc kết quả đã lưu; không chạy lại test hoặc thay model mặc định.


In [ ]:
import hashlib, json
from pathlib import Path
V4_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
v4_summary = json.loads((V4_ROOT/'outputs/stream_v4_20261025/summary.json').read_text(encoding='utf-8'))
v4_weights = V4_ROOT/'outputs/stream_v4_20261025'/v4_summary['selected_checkpoint']/'model.safetensors'
if not v4_weights.exists():
    v4_weights = V4_ROOT/v4_summary['selected_checkpoint']/'model.safetensors'
with v4_weights.open('rb') as stream:
    assert hashlib.file_digest(stream, 'sha256').hexdigest() == v4_summary['weights_sha256']
assert not v4_summary['user_target_achieved']
print(json.dumps({'hash_verified':True, 'status':v4_summary['status'],
    'fresh_natural_AUTO_labels':v4_summary['fresh_natural_auto_label_metrics'],
    'fresh_crawl_gold_labels':v4_summary['fresh_crawl_gold_labels'],
    'fresh_crawl_accuracy':None, 'serving_default_changed':False},ensure_ascii=False,indent=2))


## V5: chỉ nhận text, bổ sung hội thoại tự nhiên

Train mới1.433mẫu từ4cuộc gọi Việt,101cuộc gọi Anh,93cuộc gọi Trung; dev293mẫu.384mẫu giữ riêng để chấm sau chọn checkpoint. Dữ liệu cũ chỉ replay từ train; pilot70mẫu V4 chỉ làm regression.

ASR tạo chữ từ phần âm thanh đã quan sát. Mô hình nhận text + lịch sử chữ cũ; không nhận âm thanh/tương lai. Chia tập bằng call_id chính thức, giữ cả hai người nói của một cuộc gọi trong cùng tập. Nhãn khoảng dừng tự động, chưa có human gold.

Ô dưới đọc trạng thái job đã chạy riêng trên Colab. Có thể chạy lại ô để xem tiến độ; ô này không khởi chạy hay ngắt train. Mục tiêu >90% vẫn phải đo trên tập giữ riêng sau khi chọn model.


In [ ]:
import json
from pathlib import Path
N5_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
N5_NAME = 'natural_v5_train_20261025'
def n5_read(relative):
    for base in [N5_ROOT, N5_ROOT/'outputs/natural_v5_20261025']:
        path = base/relative
        if path.exists(): return json.loads(path.read_text(encoding='utf-8'))
    return None
n5_status = n5_read('jobs/'+N5_NAME+'/status.json')
n5_progress = n5_read('runs/'+N5_NAME+'/progress.json')
n5_selection = n5_read('runs/'+N5_NAME+'/selection.json')
n5_manifest = n5_read('data_v5/canonical_20261025/manifest.json')
print(json.dumps({'job_status':n5_status, 'progress':n5_progress,
    'best_dev': n5_selection['best_dev'] if n5_selection else None,
    'natural_dataset_counts': n5_manifest['files'] if n5_manifest else None,
    'model_input':'TEXT_ONLY', 'sealed_test_scored':bool(n5_read('data_v5/canonical_20261025/sealed_consumed.json')),
    'note':'Dev is not held-test quality. Rerun this cell to read saved job state; it never launches/restarts training.'},ensure_ascii=False,indent=2))


## V5–V8: kết quả hội thoại tự nhiên trên ba ngôn ngữ

Đã train thêm teacher, MiniLM và Qwen3-4B LoRA với đầu vào text-only. Lựa chọn model/ngưỡng đã khóa trên dev trước khi chấm 384 mẫu giữ riêng; các mẫu đó nay là regression. Nhãn tham chiếu observed-floor/VAD tự động, chưa có human gold. Bảng dưới đọc kết quả Colab đã lưu.

[Kết quả kiểm tra cuối](outputs/natural_v7_20261025/review.html) · [Phát lại 204 prefix crawl](outputs/natural_v7_20261025/crawl_review.html) · [Nghe 20 ví dụ ASR](outputs/natural_v7_20261025/asr_review/review.html) · [Recipe và giới hạn kết luận](README_NATURAL_CAMPAIGN.md).


In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

NC_ROOT=Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
def nc_read(relative):
    roots=[NC_ROOT]+[NC_ROOT/'outputs'/name for name in ['natural_v5_20261025','natural_v6_20261025','natural_v7_20261025','natural_v8_20261025']]
    for base in roots:
        path=base/relative
        if path.exists():
            try:return json.loads(path.read_text(encoding='utf-8'))
            except json.JSONDecodeError:return None
    return None

nc_jobs=[]
for name in ['natural_v5_train_20261025','natural_v6_finetune_20261025','natural_v6_pho_repair_20261025',
    'natural_v7_text_20261025','natural_v8_lora_20261025','natural_v8_lora_20261025_r2','natural_v8_lora_20261025_r3','natural_final_eval_20261025']:
    status=nc_read('jobs/'+name+'/status.json') or {}
    nc_jobs.append({'job':name,'status':status.get('status','chưa có receipt'),'exit_code':status.get('exit_code'),
        'elapsed_seconds':status.get('elapsed_seconds')})
print('Đọc trạng thái lúc UTC:',datetime.now(timezone.utc).isoformat())
display(pd.DataFrame(nc_jobs))
nc_progress=nc_read('runs/natural_v8_lora_20261025_r3/progress.json') or nc_read('runs/natural_v7_text_20261025/progress.json')
if nc_progress:print('Tiến độ train đã lưu:',json.dumps(nc_progress,ensure_ascii=False))

nc_frozen=nc_read('outputs/natural_v7_20261025/selected_before_held.json')
nc_selection=nc_read('runs/natural_v8_lora_20261025_r3/selection.json') or nc_read('runs/natural_v7_text_20261025/selection.json')
nc_dev=nc_frozen['selected']['dev'] if nc_frozen else nc_selection['best_dev'] if nc_selection else None
if nc_dev:
    print('Development — không phải điểm test giữ riêng:')
    display(pd.DataFrame([{'language':lang,'samples':m['samples'],'accuracy_pct':round(100*m['accuracy'],2),
        'F1_EOT_pct':round(100*m['eot_f1'],2),'false_EOT_pct_of_holds':round(100*m['false_eot_rate'],2)}
        for lang,m in nc_dev['natural_languages'].items()]))
nc_final=nc_read('outputs/natural_v7_20261025/final_eval/report.json')
if nc_final:
    print('Kiểm tra cuối — nhãn observed-floor/VAD tự động, chưa có human gold:')
    display(pd.DataFrame([{'language':lang,'samples':m['samples'],'accuracy_pct':round(100*m['accuracy'],2),
        'F1_EOT_pct':round(100*m['eot_f1'],2),'false_EOT_pct_of_holds':round(100*m['false_eot_rate'],2)}
        for lang,m in nc_final['metrics']['selected'].items()]))
    print('Numeric target:',nc_final['automatic_held_numeric_target_met'],'| Số cuộc gọi:',nc_final['held_calls'])
else:
    print('384 mẫu giữ riêng: chưa có kết quả classifier đã hoàn tất.')
print('Đầu vào model: TEXT_ONLY. Ô này chỉ đọc trạng thái; không khởi chạy/ngắt train. Crawl204prefix chưa có gold nên accuracy/F1 chưa xác định.')


## Ba tiếng — ba model độc lập

Giữ bản Việt QAT đã có, train teacher → distill → QAT riêng cho Anh và Trung. Mỗi tiếng có model/tokenizer/ngưỡng riêng. Điểm dev tiền tố cũ không phải chất lượng trên hội thoại mới. [Recipe và giải thích](README_MONOLINGUAL.md).


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

M_ROOT=Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
def m_read(relative):
    for base in [M_ROOT,M_ROOT/'outputs/monolingual_20261025']:
        path=base/relative
        if path.exists():return json.loads(path.read_text(encoding='utf-8'))
    return {}

print('Ba model độc lập: VI dùng bản QAT đã có; EN/ZH train teacher → distill → QAT riêng.')
print('VI cũ:93,55% accuracy /90,19% F1 trên dev tiền tố cũ; không phải điểm hội thoại mới.')
records=[]
for lang in ['en','zh']:
    for phase in ['teacher','student']:
        name='mono_'+lang+'_'+phase+'_20261025'
        status=m_read('jobs/'+name+'/status.json');progress=m_read('runs/'+name+'/progress.json')
        records.append({'language':lang,'phase':phase,'status':status.get('status','chưa bắt đầu'),
            'step':progress.get('step'),'stage':progress.get('stage',progress.get('phase')),'exit_code':status.get('exit_code')})
display(pd.DataFrame(records))
metrics=[]
for lang in ['en','zh']:
    for phase in ['teacher','student']:
        name='mono_'+lang+'_'+phase+'_20261025';report=m_read('runs/'+name+('/training_report.json' if phase=='teacher' else '/monolingual_report.json'))
        for scope,value in report.get('dev_slices',{}).items():
            metrics.append({'language':lang,'phase':phase,'scope':'DEV '+scope,'samples':value['samples'],
                'accuracy_pct':round(value['accuracy']*100,2),'F1_EOT_pct':round(value['eot_f1']*100,2)})
if metrics:
    print('Development dùng chọn model/ngưỡng — chưa là kiểm chứng độc lập:')
    display(pd.DataFrame(metrics))
final=m_read('outputs/monolingual_20261025/report.json')
if final:
    print('Regression đã xem; nhãn tự động, chưa có human semantic gold:')
    display(pd.DataFrame([{'language':lang,'samples':v['samples'],'accuracy_pct':round(v['accuracy']*100,2),
        'F1_EOT_pct':round(v['eot_f1']*100,2)} for lang,v in final['regression_metrics'].items()]))
print('Ô này chỉ đọc kết quả/trạng thái, không khởi chạy thêm job.204prefix crawl chưa có nhãn nên chưa tính accuracy/F1.')


## V10 — data mở rộng, crawl Qwen, đối chứng reward

Ba model riêng, QAT INT4. So sánh data_sft/CE/cost-CE/REINFORCE trên cùng dev. Nhãn tự động/silver, chưa có chứng nhận90% thực. [Báo cáo](outputs/reward_v10_20261025/review.html) · [Recipe](README_REWARD_V10.md).


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
V10_ROOT=Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
def v10_read(relative):
    path=V10_ROOT/relative
    if path.exists():return json.loads(path.read_text(encoding='utf-8'))
    return {}
print('Ba model riêng. V10 thêm data và thử reward một quyết định; các điểm dưới là dev, không phải test mới.')
jobs=[]
for name in ['reward_v10_campaign_20261025','reward_v10_qwen_20261025','reward_v10_recovery_20261025']+['reward_v10_'+lang+'_20261025_r2' for lang in ['vi','en','zh']]:
    state=v10_read('jobs/'+name+'/status.json');jobs.append({'job':name,'status':state.get('status','chưa bắt đầu'),'exit_code':state.get('exit_code'),'seconds':state.get('elapsed_seconds')})
display(pd.DataFrame(jobs))
progress=[];metrics=[]
for lang in ['vi','en','zh']:
    p=v10_read('outputs/reward_v10_20261025/'+lang+'/progress.json')
    if p:progress.append({'language':lang,'phase':p.get('phase'),'step':p.get('step'),'max_steps':p.get('steps_max'),'batch':p.get('batch_size')})
    report=v10_read('outputs/reward_v10_20261025/'+lang+'/report.json')
    if report:
        for phase,record in [('baseline',report['baseline'])]+[(r['phase'],r['dev']) for r in report['phases']]:
            m=record['slices']['natural'];metrics.append({'language':lang,'phase':phase,'accuracy_pct':round(100*m['accuracy'],2),'F1_EOT_pct':round(100*m['eot_f1'],2),'false_EOT_pct_of_holds':round(100*m['false_eot_rate'],2),'min_dev_slice_score_pct':round(100*record['selection_score'],2)})
if progress:display(pd.DataFrame(progress))
if metrics:display(pd.DataFrame(metrics))
qwen=v10_read('outputs/reward_v10_20261025/qwen/report.json')
if qwen:print('Qwen two-view silver:',qwen['by_language'],'| Chưa hiệu chỉnh confidence, không phải human gold.')
print('Reward: đúng+1/ngắt sớm−2/chờ trênEOT−0,5. Không có measured latency. Crawl silver không nhận policy reward. Ô này không khởi chạy/ngắt train.')


## Tự động crawl và RL — từng tiếng một model

Qwen3-8B làm người chấm semantic EOT/NEOT/AMBIGUOUS. Crawl theo vòng → chấm reward → cập nhật policy nhỏ → giữ/bỏ theo dev. Nhãn judge không phải human gold. Đã chạy2vòng giới hạn; chưa chứng nhận90%hội thoại thực. [Kết quả và số liệu](outputs/auto_rl_v12_20261025/review.html) · [Cách chạy](README_AUTO_RL.md).


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
AR_ROOT=Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
def ar_read(relative):
    path=AR_ROOT/relative
    if path.exists():return json.loads(path.read_text(encoding='utf-8'))
    return {}
print('Automatic crawl → model action → Qwen judge reward → one-step RL → dev gate; three independent policies.')
state=ar_read('jobs/auto_rl_v12_20261025/status.json');report=ar_read('outputs/auto_rl_v12_20261025/report.json')
print('Job:',state.get('status'),'| GPU memory peak GiB:',round(report.get('GPU_peak_bytes',0)/2**30,2))
print('Judge Qwen3-8B:',report.get('probe_passed'),'| reward is NOT human gold or measured turn timing.')
if report.get('rounds'):display(pd.DataFrame(report['rounds']))
rows=[]
for rnd in [1,2]:
    for lang in ['vi','en','zh']:
        r=ar_read('outputs/auto_rl_v12_20261025/round_%02d/'%rnd+lang+'/report.json')
        if r.get('dev_after'):
            before=r['dev_before']['slices']['natural'];after=r['dev_after']['slices']['natural']
            rows.append({'round':rnd,'language':lang,'accuracy_before_pct':round(100*before['accuracy'],2),'F1_before_pct':round(100*before['eot_f1'],2),
                'accuracy_after_pct':round(100*after['accuracy'],2),'F1_after_pct':round(100*after['eot_f1'],2),'kept_by_dev_gate':r['candidate_retained']})
if rows:display(pd.DataFrame(rows))
print('Development only. No old384regression tuning. No90%real certificate. This cell reads receipts; it does not start or stop training.')


## Colab — crawl và train liên tục

Chạy đến khi người dùng yêu cầu dừng, không giới hạn2vòng. SFT/RL từ checkpoint tốt nhất của từng tiếng, data tích lũy, optimizer/RNG có thể resume. Semantic silver dev và natural-floor dev báo riêng. [Dashboard](outputs/continuous_v13_20261026/review.html) · [Cách khôi phục](README_CONTINUOUS.md).


In [ ]:
import json
from pathlib import Path
import sqlite3
import subprocess
from datetime import datetime,timezone
import pandas as pd
from IPython.display import display

C13_ROOT=Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').is_dir() else Path.cwd()
C13_RUN=C13_ROOT/'outputs/continuous_v13_20261026'
print('Đọc kernel lúc UTC:',datetime.now(timezone.utc).isoformat())
print('Chiến dịch liên tục: không giới hạn số vòng, không dừng vì dev plateau hoặc silver90%.')
if (C13_RUN/'status.json').exists():
    s=json.loads((C13_RUN/'status.json').read_text(encoding='utf-8'))
    print(json.dumps({k:s.get(k) for k in ['status','stage','cycle','language','branch','step','steps','heartbeat_utc']},ensure_ascii=False))
if (C13_RUN/'state.sqlite').exists():
    c=sqlite3.connect('file:'+str(C13_RUN/'state.sqlite')+'?mode=ro',uri=True)
    counts=[dict(zip(['language','split','label','samples'],r)) for r in c.execute('SELECT language,split,label,COUNT(*) FROM samples GROUP BY language,split,label')]
    display(pd.DataFrame(counts))
    champions=[(key.split(':')[1],json.loads(value)) for key,value in c.execute("SELECT key,value FROM meta WHERE key LIKE 'champion:%'")]
    rows=[]
    for lang,r in champions:
        semantic=r['semantic_dev']['metrics'];floor=r['natural_floor_dev']
        rows.append({'language':lang,'semantic_silver_dev_samples':semantic['samples'],'semantic_accuracy_pct':round(100*semantic['accuracy'],2),'semantic_F1_pct':round(100*semantic['eot_f1'],2),
            'natural_floor_accuracy_pct':round(100*floor['accuracy'],2),'natural_floor_F1_pct':round(100*floor['eot_f1'],2),'precision':r.get('precision','FP32_LATENT')})
    if rows:display(pd.DataFrame(rows))
    c.close()
if Path('/content').exists():
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,utilization.gpu,memory.used,memory.total','--format=csv,noheader'],text=True).strip())
if (C13_RUN/'recovery/latest_snapshot.json').exists():print('Snapshot:',json.loads((C13_RUN/'recovery/latest_snapshot.json').read_text(encoding='utf-8')))
print('Semantic silver do Qwen chấm không phải human gold. Natural-floor là mục tiêu khác. Chưa chứng nhận90%hội thoại thực.')
print('Các candidate hiện học FP32 latent; chưa thay bản INT4 serving. Ô này chỉ đọc, không tạo worker mới.')


## Teacher lớn trước — VI / EN / ZH

Fine-tune ba teacher pretrained lớn độc lập, đánh giá trước khi distill. [Dashboard](outputs/teacher_first_v14_20261026/review.html).


In [ ]:
import json
from pathlib import Path
import sqlite3
import subprocess
from datetime import datetime, timezone, timedelta
import pandas as pd
from IPython.display import display

TF_ROOT = Path('/content/turn_detect_v2') if Path('/content/turn_detect_v2').exists() else Path.cwd()
TF_RUN = TF_ROOT / 'outputs/teacher_first_v14_20261026'
print('Kiểm tra lúc:', datetime.now(timezone(timedelta(hours=7))).isoformat())
print('Teacher lớn trước, ba checkpoint riêng VI/EN/ZH; chưa distill hoặc thay model INT4.')
if (TF_RUN / 'status.json').exists():
    tf_status = json.loads((TF_RUN / 'status.json').read_text(encoding='utf-8'))
    print(json.dumps({k: tf_status.get(k) for k in ['status', 'stage', 'cycle', 'language', 'step', 'steps', 'total_steps', 'loss', 'heartbeat_utc']}, ensure_ascii=False))
if (TF_RUN / 'state.sqlite').exists():
    tf_db = sqlite3.connect('file:' + str(TF_RUN / 'state.sqlite') + '?mode=ro', uri=True)
    tf_rows = []
    for tf_key, tf_value in tf_db.execute("SELECT key,value FROM meta WHERE key LIKE 'large_teacher:%'"):
        tf_report = json.loads(tf_value); tf_dev = tf_report['semantic_dev']; tf_floor = tf_report['natural_floor_dev']
        tf_rows.append({'tiếng': tf_key.split(':')[1], 'params': tf_report['identity']['parameters'],
                       'bước': tf_report['total_steps'], 'mẫu silver dev': tf_dev['samples'],
                       'silver accuracy %': round(100 * tf_dev['accuracy'], 2), 'silver F1 %': round(100 * tf_dev['eot_f1'], 2),
                       'floor accuracy %': round(100 * tf_floor['accuracy'], 2), 'floor F1 %': round(100 * tf_floor['eot_f1'], 2)})
    if tf_rows: display(pd.DataFrame(tf_rows))
    tf_db.close()
if Path('/content').exists():
    print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'], text=True).strip())
print('Silver Qwen không phải human gold; chưa chứng nhận 90% trên hội thoại thật.')
print('Teacher và optimizer đầy đủ hiện lưu trên Colab; metadata snapshot không chứa trọng số.')
print('Ô này chỉ đọc trạng thái, không tạo worker hoặc chạy lại training.')
